# Building AI News Developer Agent with Google ADK

This `Notebbook_app_001.ipynb` demonstrates:

- Setting up a new agent folder (`adk create`)
- Writing the first `agent.py`
- Adding text models and built-in tools (`google_search`, `BuiltInCodeExecutor`)
- Fine-tuning agent instructions and behavior
- Testing valid and invalid prompts

In [1]:
# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()
import os 

In [2]:
print("HF configured:", bool(os.getenv("HUGGING_FACE_TOKEN")))
print("GitHub configured:", bool(os.getenv("GITHUB_TOKEN")))


HF configured: True
GitHub configured: True


## Setting up the agent  

Run the cell below to create the folder structure for the agent.  

In [ ]:
!adk create --type=code app_01 --model gemini-2.5-flash --api_key $GEMINI_API_KEY  

- Using the `adk create` command, the below new folder structure with ADK's built-in project scaffolding is set up,  generating three essential files:  

File structure:  

```bash 
app_01/
    __init__.py   # The `__init__.py` file marks the directory as a Python package, nabling proper imports.   
    agent.py      # he `agent.py` file provides a clean foundation where you'll implement your agent.  
    .env          # The `.env` file securely stores your API credentials and configuration.
```

>_Note_ : In this the project `--type=code` option has been selected to generate a Python-based agent in `agent.py`.

- Adding a text model    

The `--model` parameter specifies the LLM to be used by the agent. Here it will be used a text-focused model like `gemini-2.5-flash` since this is ideal when the purpose is to optimize text processing and to provide faster response times.  

- SAY SOMETHING ABOUT THE GEMINI_API_KEY 

## Writing the `agent.py`

`adk create` command is used to create folders and then write to its `agent.py` using the specific command, `%%writefile FILENAME`, to interact with the files in the new agent folder. 

### Adding tools to the agent  

**But here's the problem:** try asking this agent about the latest AI developments, and you'll quickly discover it can only tell you about things that happened before its training cutoff date. For an AI news assistant that's supposed to fetch the latest news, that's not particularly helpful.

Therefore, you need to fix that by providing your agent with **Tools**. In Google ADK, the word [“tool”](https://google.github.io/adk-docs/tools/) has two meanings:  

| Term                 | Meaning                                                                   |
| -------------------- | ------------------------------------------------------------------------- |
| **ADK Tool**         | A callable object exposed to an agent (e.g. `google_search`, `AgentTool`) |
| **Third-Party Tool** | Any external system or API                                                |


#### Adding Google Search Tool

In this scenario, to fetch the latest news, let's provide the agent with a built-in tool, `google_search`. These built-in tools come pre-packed with the library. To add it to the agent, just import it and provide it as a tool in the tools array.  Let's test the agent with the Google search tool by asking a query `"What is the latest AI News?"`. The agent will use the Google Search tool to find current information, process the results, and give you a comprehensive, up-to-date response with sources. Just like that, the agent can now access **real-time information** from across the web!

ADK comes with several other `powerful built-in tools—there` are tools for running your code in a `sandbox`, `querying databases`, even `integrations with Google Workspace tools` like Calendar, Drive etc.  

#### Adding Python code executor Tool

To fetch the latest news, the agent has been provided with a built-in tool, **google_search**. These built-in tools come `pre-packed with the library`. Now, in order to allow the agent also to execute code and debug using Gemini models, it has been used the **built_in_code_execution tool** that enables the agent to execute code, specifically when using Gemini 2 and higher models. This allows the model to perform tasks like calculations, data manipulation, or running small scripts. Also this tool comes from `pre-packed with the library`. To add it to the agent, just import it and provide it as a tool in the tools array. Run the cell below to import it

#### Adding HuggingFace Execution Agent Tool  
TO BE DRAFTED   

#### Adding GitHub Execution Agent Tool  
TO BE DRAFTED     

### Fine-tuning agent instructions

So far the agent has simple instructions, but for reliable behavior, you need more sophisticated instruction engineering. Therefore, the agent has been enhanced  with strict behavioral controls.  

In [3]:
%%writefile app_01/agent.py
import os
import re
from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool
from google.adk.code_executors import BuiltInCodeExecutor

# ======================================================
# Environment variables
# ======================================================
HF_TOKEN = os.getenv("HUGGING_FACE_TOKEN")
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")


# ======================================================
# AI Developer News Agent (FIXED – NO LOOPS)
# ======================================================
search_agent = Agent(
    model="gemini-2.5-flash",
    name="AIDevSearchAgent",
    description="Specialist agent for AI developer news with structured enrichment.",
    instruction="""
You are an AI News Analyst for developers.

Scope:
- ONLY AI-related news relevant to developers.
- ALWAYS use google_search for factual information.

Rules:
- NEVER ask follow-up questions.
- If the user does not specify a number of articles, DEFAULT to 3.
- If a number is present in the request, use it as the article count.

Response format (REQUIRED):

Using google_search, here are the top headlines:

---
[NUMBER]. HEADLINE

Summary:
1–2 sentence technical summary

Tech stack:
- frameworks / languages / infrastructure OR "Not mentioned"

License:
- Open-source | Proprietary | Mixed | Not mentioned

GitHub repository:
- Repository name if explicitly referenced
- Otherwise: Not referenced

Hugging Face:
- Model / Dataset / Space name if mentioned
- Otherwise: Not mentioned

Who should care:
- ML Engineer / Backend Engineer / MLOps / Data Scientist
---

End by asking:
"Which headline would you like to explore in more detail?"
""",
    tools=[google_search],
)


# ======================================================
# Python Code Execution Agent
# ======================================================
coding_agent = Agent(
    model="gemini-2.5-flash",
    name="CodeAgent",
    description="Executes safe Python code in a sandbox.",
    instruction="""
You execute Python code safely.

Rules:
- Do NOT access the file system
- Do NOT import os, sys, subprocess, socket, or requests
- Do NOT perform network calls
- Do NOT run infinite loops
- Only return the execution result or error
""",
    code_executor=BuiltInCodeExecutor(),
)

# ============================
# Python Code Explanation Agent
# ============================
code_explain_agent = Agent(
    model="gemini-2.5-flash",
    name="CodeExplainAgent",
    description="Explains Python code without executing it.",
    instruction="""
Explain Python code clearly and safely.

Rules:
- Do NOT execute code
- Do NOT modify code
- Explain step-by-step
- Mention pitfalls or edge cases if relevant
""",
)


# ======================================================
# Hugging Face Canonical Reference Agent
# ======================================================
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters

hf_agent = Agent(
    model="gemini-2.5-flash",
    name="hugging_face_agent",
    description="Returns validated canonical Hugging Face URLs for exact IDs only.",
    instruction="""
You are a Hugging Face reference agent.

STRICT RULES:
- ONLY return a Hugging Face URL if the user provides an EXACT Hugging Face ID.
- Do NOT guess, normalize, or infer names.

If the resource does not exist, respond:
"No Hugging Face resource found for the provided identifier."

Output format (ONLY if validated):

Hugging Face URL:
https://huggingface.co/<exact_id>
""",
    tools=(
        [
            McpToolset(
                connection_params=StdioConnectionParams(
                    server_params=StdioServerParameters(
                        command="npx",
                        args=["-y", "@llmindset/hf-mcp-server"],
                        env={"HF_TOKEN": HF_TOKEN},
                    ),
                    timeout=30,
                ),
            )
        ]
        if HF_TOKEN
        else []
    ),
)


# ======================================================
# GitHub MCP Agent
# ======================================================
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPServerParams

git_agent = Agent(
    model="gemini-2.5-flash",
    name="github_agent",
    description="Read-only GitHub repository enrichment agent.",
    instruction="""
You are a GitHub repository reference agent.

Rules:
- ONLY summarize repositories explicitly mentioned by the user.
- Do NOT invent repositories.

If no repository is mentioned or found:
Respond with:
"No GitHub repository found."
""",
    tools=(
        [
            McpToolset(
                connection_params=StreamableHTTPServerParams(
                    url="https://api.githubcopilot.com/mcp/",
                    headers={
                        "Authorization": f"Bearer {GITHUB_TOKEN}",
                        "X-MCP-Toolsets": "all",
                        "X-MCP-Readonly": "true",
                    },
                ),
            )
        ]
        if GITHUB_TOKEN
        else []
    ),
)


# ======================================================
# Root Routing Agent
# ======================================================
root_agent = Agent(
    name="RootAgent",
    model="gemini-2.5-flash",
    description="Strict routing agent that delegates all work to specialists.",
    instruction="""
You are a STRICT routing agent.

Routing rules:
- AI developer news → AIDevSearchAgent
- Python execution or code explanation → CodeAgent
- Python code explanation → CodeExplainAgent
- Hugging Face canonical links → hugging_face_agent
- GitHub repositories or activity → github_agent

Rules:
- ALWAYS delegate
- NEVER answer directly
- Ask for clarification only if intent is unclear
""",
    tools=[
        AgentTool(agent=search_agent),
        AgentTool(agent=coding_agent),
        AgentTool(agent=code_explain_agent),
        AgentTool(agent=hf_agent),
        AgentTool(agent=git_agent),
    ],
)


# ======================================================
# Helper functions
# ======================================================
def extract_headlines(response_text: str):
    return re.findall(r"\d+\.\s*(.+)", response_text)


def extract_limit(text: str, default: int = 3) -> int:
    match = re.search(r"\b(\d+)\b", text)
    return int(match.group(1)) if match else default


def handle_user_input(user_input: str, session_state: dict):
    if session_state is None:
        session_state = {"headlines": []}

    headlines = session_state.get("headlines", [])

    # Exit
    if user_input.lower() in {"exit", "quit"}:
        return "Goodbye!", {"headlines": []}

    # Explicit Python execution
    if user_input.lower().startswith("execute python code:"):
        code = user_input[len("execute python code:"):].strip()
        try:
            result = coding_agent.run(code)
            return f"**Code Result:**\n{result}", session_state
        except Exception as e:
            return f"Execution error: {e}", session_state

    # Explicit Python code explanation

    if user_input.lower().startswith("explain this python code:"):
        code = user_input[len("explain this python code:"):].strip()
        return code_explain_agent.run(code), session_state

    # Headline selection
    if user_input.isdigit() and headlines:
        idx = int(user_input) - 1
        if 0 <= idx < len(headlines):
            headline = headlines[idx]

            summary = search_agent.run(
                f"Provide a deeper technical summary for: {headline}"
            )

            github_info = git_agent.run(
                "Summarize the GitHub repository mentioned, if any."
            )

            hf_info = hf_agent.run(
                "Return the Hugging Face URL if an exact ID is mentioned."
            )

            return (
                f"{summary}\n\n---\n"
                f"GitHub enrichment:\n{github_info}\n\n"
                f"Hugging Face enrichment:\n{hf_info}"
            ), session_state

    # Default routing (news count handled by agent)
    response = root_agent.run(user_input)
    session_state["headlines"] = extract_headlines(response)
    return response, session_state


Overwriting app_01/agent.py
